# Chapter 10: Weak Supervision

This notebook accompanies **Chapter 10** of the lecture notes.

> Today there is *some* supervision, but it is partial, noisy, and conflicting. A heuristic rule with high precision on a few cases. A crowd annotator who flips occasionally. A pretrained classifier that is confidently wrong outside its training distribution. The chapter abstracts all three into a single object, the *labelling function*, and asks how to train a model when nobody fully trusts any single source.

**Agenda**

🛠️ · 🧮 · 🎓 · ⚖️ · 🏁

**Take it from here:** 🔍 · 🪜


In [ ]:
import numpy as np
import sys; sys.path.insert(0, '../..')
from plot_style import *
from sklearn.linear_model import LogisticRegression
from checks import (
    check_lf_top_loop,
    check_lf_coverage, check_lf_empirical_accuracy,
    check_majority_vote,
    check_soft_cross_entropy, check_pick_most_uncertain,
    ABSTAIN,
)
from viz_helpers import (
    load_binary_digits, apply_lfs,
    lf_noisy_crowd, lf_left_loop_edge, make_judge,
    em_label_model,
    train_logistic_on_soft, predict_proba,
    run_active_learning, run_random_baseline,
    plot_active_learning_curve, show_lf_examples,
    show_task_hardness, show_label_model_predictions,
    plot_label_matrix, plot_coverage_accuracy,
    plot_em_label_model_trace, plot_em_soft_pos_evolution,
    plot_soft_posteriors,
    plot_verifier_generator_gap,
    show_active_learning_picks,
)

RNG = np.random.default_rng(0)


## 🛠️ Labelling Functions

A *labelling function* (LF) is a Python function that takes an input and returns either a label or `ABSTAIN`. We work on a binary task: classify 8×8 sklearn digits as **4 vs 9** (label `0` = 4, `1` = 9). The training pool is unlabelled; a 20-example *dev set* is the only labelled data we have, used to read off each LF's accuracy.

The intro listed three kinds of weak source — heuristic, crowd, pretrained classifier. We instantiate the first two as cheap LFs here, and defer the third to §⚖️, where it returns in a different role: a *judge*, a stronger learned model bolted onto the same interface. Until then, the three LFs are all cheap, all making *different* mistakes — the property the label model exploits:
- **`lf_top_loop`**: a hand-crafted heuristic. A 9 has a closed loop at the top, a 4 has an open top.
- **`lf_noisy_crowd`**: a simulated crowd annotator. Knows the truth most of the time, flips with probability 0.2, abstains on 30% of the data.
- **`lf_left_loop_edge`**: a second hand-crafted heuristic on a *different* image region. A 9's loop has a left edge that sits in cols 1-2; a 4's left vertical is more central and leaves cols 1-2 dimmer. Its mistakes are largely uncorrelated with `lf_top_loop`'s, which is what gives the label model its triangulation signal.


In [ ]:
X, y, X_test, y_test, X_dev, y_dev = load_binary_digits(seed=0)
print(f'Training pool: {X.shape}  (no labels)   Test: {X_test.shape}   Dev: {X_dev.shape}')

# A look at the task: real 4s and 9s with class means in the right column.
# The difference between the two classes lives almost entirely in the top
# few rows of the image. That's the structure your LF will exploit.
show_task_hardness(X, y, n_each=6)


### Your first LF: read off the top loop

The class-mean panel above is the most useful thing on the page. A 9 has a closed loop in rows 0 and 1, a 4 has an open top, and the rest of the image carries far less signal. Write `lf_top_loop(X)` that turns this observation into a vote.

> An LF must decide between *vote* and *ABSTAIN*. What does picking a band of "I don't know" buy you that a hard threshold doesn't?

<details><summary>Thought</summary>

The label model in §🧮 will weight each LF by its accuracy on the rows where it fires. Forcing a vote on every row drags the LF's accuracy down, because confused middle examples count against it. A wide abstain band trades coverage for precision; the label model can recover the lost coverage from the other LFs. This is the precision/coverage trade-off you'll see plotted below.

</details>

`X` has shape `(n_samples, 64)`; reshape to `(n, 8, 8)` to read pixel regions. Pick a region whose mean intensity differs between the two classes, threshold it twice (one threshold per class), and return ABSTAIN in the middle. Output shape `(n_samples,)`, values in `{0, 1, ABSTAIN}` (recall: label `0` = digit 4, label `1` = digit 9).


In [ ]:
def lf_top_loop(X):
    """Hand-crafted heuristic, intentionally suboptimal to start with.

    The top-centre region (rows 0:2, cols 2:6) is filled in a 9 (closed loop)
    and dim in a 4 (open top). Class means on this dataset: 4 ~ 0.38, 9 ~ 0.62.
    The thresholds below leave only a narrow abstain band (0.42 to 0.45), so
    the LF fires on most rows but accepts confused middle examples it should
    have abstained on. Compare with a wider band (e.g. 0.32 / 0.55) to feel
    the precision/coverage trade-off: tighter band -> higher coverage, lower
    accuracy; wider band -> higher accuracy, lower coverage.

    Parameters
    ----------
    X : ndarray of shape (n_samples, 64), pixel intensities in [0, 1]

    Returns
    -------
    ndarray of shape (n_samples,), values in {0, 1, ABSTAIN}.
    """
    images = X.reshape(-1, 8, 8)
    top_centre = images[:, 0:2, 2:6].mean(axis=(1, 2))
    out = np.full(len(X), ABSTAIN, dtype=int)
    out[top_centre > 0.45] = 1   # label 1 = digit 9
    out[top_centre < 0.42] = 0   # label 0 = digit 4
    return out


check_lf_top_loop(lf_top_loop)


In [ ]:
# See where your LF fires on the pool.
if lf_top_loop(X) is not None:
    out_lead = lf_top_loop(X)
    show_lf_examples(X, y, out_lead, 'lf_top_loop', n_each=4)


**Coverage** = fraction of inputs the LF didn't abstain on. **Accuracy** (on the dev set) = of the times it fired, how often it was right. The two together describe an LF's precision/coverage trade-off.

Implement `lf_coverage(L)` and `lf_empirical_accuracy(L_dev, y_dev)`. `L` has shape `(n_samples, n_lfs)` with values in `{0, 1, ABSTAIN}`.


In [ ]:
def lf_coverage(L):
    """Per-LF fraction of non-abstain entries.

    Parameters
    ----------
    L : ndarray of shape (n_samples, n_lfs), values in {0, 1, ABSTAIN}

    Returns
    -------
    ndarray of shape (n_lfs,), values in [0, 1]
    """
    return (L != ABSTAIN).mean(axis=0)


check_lf_coverage(lf_coverage)

In [ ]:
def lf_empirical_accuracy(L_dev, y_dev):
    """Per-LF accuracy on (L_dev, y_dev), excluding abstain rows.

    Parameters
    ----------
    L_dev : ndarray of shape (n_dev, n_lfs)
    y_dev : ndarray of shape (n_dev,) with the true labels

    Returns
    -------
    ndarray of shape (n_lfs,), with NaN if the LF abstained on every dev example.
    """
    n_lfs = L_dev.shape[1]
    out = np.empty(n_lfs)
    for i in range(n_lfs):
        mask = L_dev[:, i] != ABSTAIN
        out[i] = (L_dev[mask, i] == y_dev[mask]).mean() if mask.any() else float("nan")
    return out


check_lf_empirical_accuracy(lf_empirical_accuracy)

### The 3-LF stack

We build the label matrix `L` once. The crowd LF needs ground truth (it's a simulated annotator), so we apply it per split with the corresponding labels and column-stack with the pure-input LFs. Every LF ends up in the same `(n_samples, n_lfs)` matrix; downstream nothing knows where each column came from.


In [ ]:
LF_NAMES = ['top_loop', 'noisy_crowd', 'left_loop_edge']

def build_L(Xs, ys, seed):
    return np.column_stack([
        lf_top_loop(Xs),
        lf_noisy_crowd(Xs, ys, flip_rate=0.20, abstain_rate=0.30, seed=seed),
        lf_left_loop_edge(Xs),
    ])

L      = build_L(X,     y,     seed=0)
L_dev  = build_L(X_dev, y_dev, seed=1)
print(f'L: {L.shape}, values in {{0, 1, -1}}')

if lf_coverage(L) is not None and lf_empirical_accuracy(L_dev, y_dev) is not None:
    cov = lf_coverage(L)
    acc = lf_empirical_accuracy(L_dev, y_dev)
    print(f'\n{"LF":<16}  coverage  accuracy')
    for n, c, a in zip(LF_NAMES, cov, acc):
        a_str = f'{a:.2f}' if not np.isnan(a) else '   nan'
        print(f'{n:<16}    {c:.2f}      {a_str}')


### The label matrix, in pictures

The table above is the entire pipeline's input from this point on. Every downstream method (majority vote, the EM label model, the end model) sees only `L`, a `(n_samples, n_lfs)` matrix of votes and abstains. The picture below is the same matrix as a categorical heatmap, sorted top-to-bottom by how many LFs fired on each row. Row 0 has all three LFs voting; the rows near the bottom have one or zero votes, which are the rows the end model will have to learn to label without help.

In [ ]:
if lf_coverage(L) is not None:
    plot_label_matrix(L, LF_NAMES, max_rows=80)
else:
    print('⬜ Need lf_coverage implemented to render the label matrix.')


### The precision/coverage trade-off

Every LF lives somewhere on a coverage/accuracy plane. A perfect-but-narrow heuristic sits in the upper-left (high accuracy, low coverage); a noisy-but-broad annotator sits in the upper-right at lower accuracy; a useless one sits near the random line at 0.5. The whole point of stacking LFs is that no single point on this plane is enough, but several together can be aggregated into something that has both high coverage and high accuracy.

In [ ]:
if (lf_coverage(L) is not None
    and lf_empirical_accuracy(L_dev, y_dev) is not None):
    plot_coverage_accuracy(lf_coverage(L),
                           lf_empirical_accuracy(L_dev, y_dev),
                           LF_NAMES)
else:
    print('⬜ Need lf_coverage and lf_empirical_accuracy implemented.')


## 🧮 Label Model

Three LFs, conflicting votes, no ground truth on the training pool. We need one label per example. The naive aggregation is **majority vote**: count the non-abstain votes for each class and pick whichever class has more. Two cases force majority vote to **abstain on a row**: (a) every LF abstained on it, so there are no votes to count, and (b) the votes are evenly split (one vote for 0 and one for 1, or two-vs-two), so neither class wins. With three LFs and a binary task this happens often: any row where one LF votes 0, one votes 1, and the third abstains is a tie.

The principled aggregation is an **agreement-based label model**, fitted by **expectation-maximisation (EM)**: estimate each LF's accuracy from agreement structure alone, then weight votes by inferred accuracy. EM also makes a meaningful decision on tie rows: a confident vote from a high-alpha LF outweighs an opposing vote from a low-alpha LF, so the soft posterior is rarely exactly 0.5.

> The label model has no labelled data. What lets it tell a good LF from a bad one?

<details><summary>Thought</summary>

Agreement is the signal. If two LFs agree on 80% of the rows where both fire, the model that calls them both 50% accurate predicts 50% agreement; calling them both 80% predicts 0.8² + 0.2² = 68%; 90% predicts 82%. The likelihood of the observed agreement pattern is highest at the right accuracies, and EM climbs that likelihood.
</details>

In [ ]:
def majority_vote(L):
    """Naive aggregation: per-row majority of non-abstain votes; ABSTAIN on ties.

    Parameters
    ----------
    L : ndarray of shape (n_samples, n_lfs)

    Returns
    -------
    ndarray of shape (n_samples,), values in {0, 1, ABSTAIN}.
    """
    n0 = (L == 0).sum(axis=1)
    n1 = (L == 1).sum(axis=1)
    out = np.full(L.shape[0], ABSTAIN, dtype=int)
    out[n1 > n0] = 1
    out[n0 > n1] = 0
    return out


check_majority_vote(majority_vote)

### What the Expectation maximization label model does (intuitively)

We hand the label matrix `L` to `em_label_model(L, n_iters)` and get back two things:
- `alphas`: one number per LF, the model's estimate of how often that LF votes correctly when it fires
- `soft_pos`: one number per training example, `P(y = 1 | L)`

Internally it alternates two steps. First, **score each example**: every LF's vote contributes evidence proportional to its current accuracy estimate, abstains contribute nothing, and a sigmoid of the total log-odds gives the row's soft posterior. Second, **rescore each LF**: how often does its non-abstain vote agree with the current soft posterior? That agreement is the new accuracy estimate. Each step makes the other look more sensible, and after a few iterations the whole thing settles. No labelled data ever enters the loop; the only signal driving it is the agreement structure of `L`.

We initialise every LF at `alpha = 0.7`, a *flat prior*: no opinion yet about which LF to trust. The plots below show how the loop moves from there.

In [ ]:
# Aggregate the label matrix and compare on the held-out test set.
alphas, soft_pos = em_label_model(L, n_iters=30)
mv_pool          = majority_vote(L)

acc_emp = lf_empirical_accuracy(L_dev, y_dev)
print('Per-LF accuracies (recovered from agreement structure of L; dev = empirical on labelled dev set):')
print(f'  {"LF":<16} {"EM":>6} {"dev":>6}')
for n, a, e in zip(LF_NAMES, alphas, acc_emp):
    e_str = f'{e:.2f}' if not np.isnan(e) else '   nan'
    print(f'  {n:<16} {a:>6.2f} {e_str:>6}')

# Apply both aggregators to the test-set label matrix L_test.
L_test       = build_L(X_test, y_test, seed=2)
mv_test      = majority_vote(L_test)
_, soft_test = em_label_model(L_test, n_iters=30)

keep   = mv_test != ABSTAIN
mv_acc = float((mv_test[keep] == y_test[keep]).mean()) if keep.any() else float('nan')
em_acc = float(((soft_test > 0.5).astype(int) == y_test).mean())

print(f'\nAggregating L_test ({len(y_test)} test rows) into one label per row:')
print(f'  {"method":<18} {"accuracy":>10} {"coverage":>14}')
print(f'  {"majority_vote":<18} {mv_acc:>10.0%} {f"{keep.sum()}/{len(y_test)}":>9} (rest abstain)')
print(f'  {"EM label model":<18} {em_acc:>10.0%} {f"{len(y_test)}/{len(y_test)}":>9} (every row gets a soft posterior)')
print('\nMV accuracy is conditioned on its non-abstain rows (denominator above);')
print('EM accuracy uses argmax of the soft posterior on every row.')


### MV vs EM on individual test rows

The numbers above hide the per-example story. The grid below picks a few test images and shows what each aggregator returned. Top row: examples majority vote refused to label, where EM is the only signal. Bottom row: examples majority vote did vote on. Each title shows the true class, the majority-vote output (or `abstain`), and the EM soft posterior `P(y = 1 | L)`. Title colour is teal when EM's argmax matches the truth, terra when it doesn't.


In [ ]:
show_label_model_predictions(X_test, y_test, mv_test, soft_test, n_each=4)


### EM converging from a flat prior

We initialise every LF at the same inferred accuracy (alpha = 0.7), the *flat prior*: the model has no opinion yet about which LF to trust. EM then alternates the E-step and M-step from the previous cell, and the two reinforce each other. A higher alpha for an LF pulls the soft posterior toward that LF's votes, which then pulls the LF's measured agreement (and so its alpha) up further. An LF that disagrees more often than the rest gets pulled the other way. The only signal driving the curves apart is the agreement structure of `L`; there is no labelled training data anywhere in the loop. After a handful of iterations the LFs separate into the order their dev accuracies suggest.

**Why we don't initialise at 0.5.** Setting every alpha to 0.5 is a fixed point EM has no way to leave on its own. With every alpha at 0.5, the log-odds contribution of any vote is `log(0.5 / 0.5) = 0`, so every soft posterior comes out exactly 0.5. The M-step then asks how often each LF agrees with that uniform posterior, and because the posterior carries no signal the answer is again exactly 0.5 for every LF. Both steps return the numbers they got, and the loop sits there forever. It is a *saddle*, not a true optimum: the data-explaining solution sits elsewhere on the likelihood surface, but from 0.5 every direction looks flat. Starting at 0.7 breaks the symmetry. An LF that is genuinely better than random nudges the posterior in its preferred direction, the M-step rewards it with a slightly higher alpha, and EM rolls toward the data-explaining solution. The clamp `[0.5 + 1e-3, 1 - 1e-3]` in the M-step keeps a noisy LF from being pulled back across the saddle.

**Reading the plot.** The solid line per LF (one colour each) is EM's iteration by iteration estimate of that LF's accuracy (alpha). The dashed horizontal line in the matching colour is the same LF's empirical accuracy on the held-out dev set: the target EM is trying to recover without ever seeing those labels. The faint horizontal line at 0.5 is the random baseline.

In [ ]:
plot_em_label_model_trace(L, LF_NAMES, n_iters=30,
                          dev_accuracies=lf_empirical_accuracy(L_dev, y_dev))


### EM seen from the example side

The previous plot is "what EM thinks of each LF" over iterations. The plot below is "what EM thinks of each example" over the same iterations.

**How to read the four panels.** Each panel is a histogram over the training pool. The horizontal axis is the soft posterior `P(y = 1 | L)` per example, ranging from 0 ("the LFs say class 0") to 1 ("the LFs say class 1"). The vertical axis is how many examples land in each bin. The same examples are shown in every panel; what changes between panels is what EM has decided about them by that iteration. Colour shows the held-back ground truth (terra = true class 0, accent = true class 1) so we can audit whether EM is moving examples in the right direction. The dashed line at 0.5 is the argmax threshold: anything to its left becomes a class-0 prediction, anything to its right a class-1 prediction.

**What you should see.**
- **Iter 0** is the *flat prior*: every alpha = 0.7 hasn't had a chance to do anything yet, so every example sits in one tall stack at 0.5. EM has no opinion about any single example yet.
- **Iter 2** is one or two passes in. The mass starts to spread; some easy examples (every LF voting the same way) have already slid toward 0 or 1, but a wide middle band remains.
- **Iter 5** is mostly settled. Class 0 (terra) clusters near the left; class 1 (accent) clusters near the right.
- **Iter 30** is the converged label model. Two well-separated modes near 0 and 1, with a thin uncertain band in the middle (the rows where the LFs genuinely disagreed). These are exactly the rows the active-learning extension in §🔍 would pick first.

The two plots are the same EM fit, side by side: the alpha trace shows it from the *LF* side (recovered accuracies converging), this one shows it from the *example* side (soft posteriors sharpening). Together they're the picture of what `em_label_model(L, n_iters)` is doing inside.

In [ ]:
plot_em_soft_pos_evolution(L, y, n_iters=30, snapshots=(0, 2, 5, 30))


### What the label model produces

The output of the EM label model is one number per training example: the soft posterior `P(y = 1 | L)`. Split the histogram by the (held-back) ground truth and you can see the label model behaving correctly. Most of class 0's mass sits near 0, most of class 1's near 1, with a thin uncertain middle band where the LFs disagreed. Examples in that middle band are exactly what an active learner would want to ask the oracle about (we'll come back to this in §🔍).

In [ ]:
plot_soft_posteriors(soft_pos, y)


## 🎓 End Model

The label model gave us a soft posterior per training example. Train a logistic regression on the *soft* labels: same shape as cross-entropy, but the target is a probability rather than a one-hot. This is the chapter's *distillation* step. The label model is the teacher, the end model is the student, the soft posterior is the temperature-softened target. The end model then generalises to inputs no LF fired on, because it operates on the input features rather than on LF outputs.


In [ ]:
def soft_cross_entropy(soft_labels, model_probs):
    """Binary cross-entropy with soft targets.

    Parameters
    ----------
    soft_labels : ndarray of shape (n,), in [0, 1] (target P(y = 1))
    model_probs : ndarray of shape (n,), in [0, 1] (predicted P(y = 1))

    Returns
    -------
    float, the per-sample-averaged cross-entropy.
    """
    eps = 1e-7
    p = np.clip(model_probs, eps, 1 - eps)
    return float(-np.mean(soft_labels * np.log(p) + (1 - soft_labels) * np.log(1 - p)))


check_soft_cross_entropy(soft_cross_entropy)

In [ ]:
_, soft_pos_train = em_label_model(L, n_iters=30)

if soft_cross_entropy(np.array([0.5]), np.array([0.5])) is not None:
    w_end, b_end = train_logistic_on_soft(X, soft_pos_train, soft_cross_entropy,
                                           n_iters=80, seed=0)
    end_acc = float(((predict_proba(X_test, w_end, b_end) > 0.5).astype(int) == y_test).mean())
    print(f'End model (LR on soft labels)        : {end_acc:.2%}')
    print(f'LR on FULL ground truth (oracle)     : '
          f'{LogisticRegression(max_iter=200).fit(X, y).score(X_test, y_test):.2%}')

    # Verifier-generator gap: examples on which the FEWEST LFs fired.
    n_votes = (L != ABSTAIN).sum(axis=1)
    sparse_idx = np.argsort(n_votes)[:20]
    sparse_acc = float(((predict_proba(X[sparse_idx], w_end, b_end) > 0.5).astype(int)
                        == y[sparse_idx]).mean())
    print(f'\nEnd-model accuracy on the 20 sparsest-coverage rows: {sparse_acc:.2%}')
    print('The end model labels rows the LFs barely covered; '
          'verifier-generator gap in action.')


### The verifier/generator gap, by coverage bucket

The bar chart bins the training pool by how many LFs fired on each row, and asks the trained end model how well it does on each bin. The point of interest is the leftmost bar: accuracy on rows where *zero* LFs fired. The label model could not assign a useful soft target there (every column was `ABSTAIN`), so the only signal those rows received was the gradient leaked in from neighbouring rows during end-model training. The model still classifies them, because it is generalising in feature space.

In [ ]:
if soft_cross_entropy(np.array([0.5]), np.array([0.5])) is not None:
    plot_verifier_generator_gap(L, X, y,
                                lambda Xq: predict_proba(Xq, w_end, b_end))


## ⚖️ Adding a Judge

This is where the *pretrained classifier* category from the intro returns — but in a different role. So far every LF in the stack is either a hand-crafted heuristic or a noisy human stand-in: cheap, weak, plentiful. A *judge* is the move that bolts a *strong* learned model onto the same LF interface, alongside the cheap ones, and lets the label model decide how much to weight it. In modern practice the strong model is typically a large language model invoked at inference time — the *LLM-as-judge* pattern — but the construction works with any sufficiently strong scorer.

From the aggregation pipeline's perspective the judge is just another LF: same `(input → vote ∪ abstain)` interface, slots into the same `L` matrix. EM doesn't know one column is "the judge"; it just notices that this column agrees with the truth-revealing structure of the others more often than they agree among themselves, and assigns it a higher alpha.

Pyodide can't host an LLM, so the judge here is a 3-layer multi-layer perceptron (MLP) trained on the full dev set. **The MLP is not pretending to be an LLM** — it's a stand-in for the strong-model role at toy scale. The pedagogical point is the *interface*: anything you'd want to do with an LLM-judge in production (override conflicts, label rows the cheap LFs missed, etc.) is wired up exactly the same way as the cell below.


In [ ]:
# Build the judge and add it as a fourth column of the label matrix.
judge_lf = make_judge(X_dev, y_dev)

L_j        = np.column_stack([L, judge_lf(X)])
LF_NAMES_J = LF_NAMES + ['judge']

L_test_j         = np.column_stack([build_L(X_test, y_test, seed=2), judge_lf(X_test)])
alphas_j, soft_j = em_label_model(L_j, n_iters=30)
print(f'{"LF":<16}  recovered accuracy')
for n, a in zip(LF_NAMES_J, alphas_j):
    print(f'{n:<16}     {a:.2f}')

w_j, b_j = train_logistic_on_soft(X, soft_j, soft_cross_entropy, n_iters=80, seed=0)
end_acc_j = float(((predict_proba(X_test, w_j, b_j) > 0.5).astype(int) == y_test).mean())
print(f'\nEnd-model accuracy with judge: {end_acc_j:.2%}')


### 🏁 Recap

- 🛠️ Three weak supervision sources (two hand-crafted heuristics on different image regions, a noisy crowd) collapsed to a single object: a function returning a vote or `ABSTAIN`.
- 🧮 EM recovered per-LF accuracies from agreement structure alone, no ground truth needed; aggregated soft posterior per example.
- 🎓 The end model trained on soft labels generalised to inputs no LF fired on; the verifier/generator gap.
- ⚖️ A stronger learned model (LLM-as-judge in production; an MLP at toy scale) slotted into the same pipeline as a fourth LF; the label model re-weighted, and the end model retrained.

The next chapter picks up the loop in a different mode: the model labels its own data and trains on its own predictions, with calibration and confirmation bias as the central failure modes.


## Take It from Here, Next Steps

### 🔍 Active learning, the opposite axis

Weak supervision sacrifices per-label *quality* for coverage; active learning sacrifices coverage for per-label *quality*. The two are complementary: cheap noisy labels scale the training set, a small budget of clean labels directs the model where it most needs them.

The simplest criterion is *uncertainty sampling*: ask the oracle about the example whose predicted `P(y=1)` is closest to 0.5.


In [ ]:
def pick_most_uncertain(model_probs):
    """Index of the entry whose predicted probability is closest to 0.5.

    Parameters
    ----------
    model_probs : ndarray of shape (n,) in [0, 1]

    Returns
    -------
    int
    """
    return int(np.argmin(np.abs(np.asarray(model_probs) - 0.5)))


check_pick_most_uncertain(pick_most_uncertain)